In [2]:
#testing of window functions self:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

pd.DataFrame({
    'id':         [1, 2, 3, 4, 5, 6],
    'name':       ['a', 'b', 'c', 'd', 'e', 'f'],
    'department': ['engineering', 'marketing', 'engineering', 'hr', 'marketing', 'engineering'],
    'salary':     [95000, 45000, 50000, 79000, 44000, 90000],
    'age':        [45, 22, 27, 29, 19, 31]
}).to_sql('employees', conn, index = False)
print('ready')


ready


In [7]:
result = pd.read_sql_query("""
    SELECT name, department, salary
    FROM (
        SELECT name, department, salary,
            ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn
        FROM employees
        ) ranked
        WHERE rn = 1
""", conn)
print(result)

  name   department  salary
0    a  engineering   95000
1    d           hr   79000
2    b    marketing   45000


In [9]:
result = pd.read_sql_query("""
    SELECT name, salary, department,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank,
            CASE
                WHEN salary > AVG(salary) OVER (PARTITION BY department)
                THEN 'above'
                ELSE 'below'
            END as vs_dept_avg
    FROM employees
    ORDER by department, dept_rank
""", conn)
print(result)
    

  name  salary   department  dept_rank vs_dept_avg
0    a   95000  engineering          1       above
1    f   90000  engineering          2       above
2    c   50000  engineering          3       below
3    d   79000           hr          1       below
4    b   45000    marketing          1       above
5    e   44000    marketing          2       below
